In [1]:
from google.colab import drive, userdata
import os

drive.mount('/content/drive')

projectPath = '/content/drive/MyDrive/TinyTransformer'
if not os.path.exists(projectPath):
  os.mkdir(projectPath)
%cd "$projectPath"

try:
  git_token = userdata.get('GITHUB_TOKEN')
except:
  print("Không lấy được TOKEN.")

!git config --global user.email "hhungln28@gmail.com"
!git config --global user.name "Hoang Hung"
!git branch -M main

username = "HoangHungLN"
repository = "TinyTransformer"
remote_url = f"https://{git_token}@github.com/{username}/{repository}.git"
!git remote set-url origin {remote_url}

Mounted at /content/drive
/content/drive/MyDrive/TinyTransformer


In [2]:
%%writefile config.py
class TinyConfig:
  def __init__(self):
    #Hardware constrants
    self.embed_dim = 64
    self.num_heads = 2
    self.depth = 3
    self.ffn_mul = 2
    self.num_classes = 10
    self.max_seq_len = 64

    #Image params
    self.img_size = 64
    self.patch_size = 8
    self.in_channels = 3

    #Audio params
    self.audio_features = 40
    self.max_audio_len = 64

Overwriting config.py


In [3]:
%%writefile embeddings.py
import torch
from torch import nn

class PatchEmbed(nn.Module):
  """ Xử lý ảnh 64x64x3 """
  def __init__(self, cfg):
    super().__init__()
    #[HW mapping]: Có thể dùng bộ MAC của Transformer accel hoặc khối Conv2d accel riêng hoặc DMA thông minh.
    self.proj = nn.Conv2d(cfg.in_channels, cfg.embed_dim, cfg.patch_size, cfg.patch_size)

  def forward(self, x):
    #[Shape] x: [B, 3, 64, 64] -> [B, 64, 8, 8] -> Flatten -> [B, 64, 64]
    x = self.proj(x).flatten(2).transpose(1, 2)
    return x

class AudioEmbed(nn.Module):
  """ Xử lý Audio Features (MFCC/Mel) """
  def __init__(self,cfg):
    super().__init__()
    #[HW mapping]: Dùng MAC của Transformer accel
    self.proj = nn.Linear(cfg.audio_features, cfg.embed_dim)

  def forward(self, x):
    #Shape x: [B, Time, Features] -> [B, Time, Embed_Dim]
    x = self.proj(x)
    return x

Overwriting embeddings.py


In [4]:
%%writefile layers.py
import torch
import torch.nn as nn

class multiHeadAttention(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.embed_dim = cfg.embed_dim
    self.num_heads = cfg.num_heads
    self.head_dim = cfg.embed_dim // cfg.num_heads
    self.scale = self.head_dim ** -0.5

    # Gom Wq, Wk, Wv vào chung một tensor để tối ưu việc load dữ liệu từ SRAM
    self.qkv = nn.Linear(cfg.embed_dim, cfg.embed_dim * 3, bias = False)
    self.proj = nn.Linear(cfg.embed_dim, cfg.embed_dim)

  def forward(self, x):
    B, N, C = x.shape
    #B: số batch, N: số token(NLP)/patch(Vision)/frame(Audio), C = embed_dim

    #Tính Q, K, V
    #[HW mapping]: Load x từ Buffer, Load W_qkv từ SRAM -> (đưa vào) Systolic Array
    qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
    #[Shape]: [B, N, C(embed_dim)] -(qkv)-> [B, N, embed_dim*3] -(reshape)-> [B, N, 3, num_heads, head_dim]
                                                              # -(permute)-> [3, B, num_heads, N, head_dim]
    q, k, v = qkv[0], qkv[1], qkv[2]
    #[Shape] q|k|v: [B, num_heads, N, head_dim]

    #attn_score = Softmax((Q * K^T)/sqrt(embed_dim))
    #attn = (Q * K^T)/sqrt(embed_dim)
    attn = (q @ k.transpose(-2, -1)) * self.scale
    #[Shape] attn: [B, num_heads, N, N]

    #Đưa qua hàm Softmax
    #[HW mapping]: Hiện thực Softmax với exp(x) xấp xỉ dùng LUT hoặc xấp xỉ đa thức
    attn = attn.softmax(dim = -1)

    #Tính attn_score * V
    x = (attn @ v).transpose(1, 2).reshape(B, N, C)
    #[Shape] x: [B, num_heads, N, head_dim] -(transpose)-> [B, N, num_heads, head_dim] -(reshape)-> [B, N, C]
                                                                                    #(C = num_heads*head_dim)

    x = self.proj(x)
    #[Shape] x: [B, N, C(embed_dim)]
    return x

class Mlp(nn.Module):
  """ Feed-Forward Network.
  [HW Mapping]: Tái sử dụng khối MMU đã dùng cho Attention.
  """
  def __init__(self, cfg):
    super().__init__()
    hidden_features = int(cfg.embed_dim * cfg.ffn_mul)

    #[HW Mapping]: lấy Weights từ SRAM và dùng MAC cho fully connected
    #[HW Mapping]: Hiện thực hàm ReLU
    self.fc1 = nn.Linear(cfg.embed_dim, hidden_features)
    self.act = nn.ReLU()
    self.fc2 = nn.Linear(hidden_features, cfg.embed_dim)

  def forward(self, x):
    x = self.fc1(x)
    x = self.act(x)
    x = self.fc2(x)
    return x

class LayerNorm(nn.Module):
  """
  LayerNorm chuẩn.
  [HW Mapping]: Vector ALU.
  Lưu ý: Để tối ưu phần cứng, có thể thay thế phép chia căn bậc hai (sqrt)
  bằng phép dịch bit (bit-shift) nếu ép phương sai về lũy thừa của 2,
  hoặc dùng bảng tra Inverse Square Root (Fast InvSqrt).
  """
  def __init__(self, normalized_shape, eps = 1e-5):
    super().__init__()
    self.weight = nn.Parameter(torch.ones(normalized_shape)) #Gamma
    self.bias = nn.Parameter(torch.zeros(normalized_shape)) #Beta
    self.eps = eps
    self.normalized_shape = normalized_shape

  def forward(self, x):
    u = x.mean(dim = -1, keepdim = True)
    s = (x - u).pow(2).mean(dim = -1, keepdim = True)
    x = (x - u) / torch.sqrt(s + self.eps)
    x = x * self.weight + self.bias

    return x

Overwriting layers.py


In [5]:
%%writefile core.py
import torch
import torch.nn as nn
from layers import multiHeadAttention, Mlp, LayerNorm

class encoder(nn.Module):
  # Pre-normalization
  def __init__(self, cfg):
    super().__init__()
    self.norm1 = LayerNorm(cfg.embed_dim)
    self.attn = multiHeadAttention(cfg)
    self.norm2 = LayerNorm(cfg.embed_dim)
    self.mlp = Mlp(cfg)

  def forward(self, x):
    x = x + self.attn(self.norm1(x))
    x = x + self.mlp(self.norm2(x))
    return x

class tinyTransformer(nn.Module):
  """
  Hardware: Accelerator Core.
  """
  def __init__(self, cfg):
    super().__init__()
    self.pos_embed = nn.Parameter(torch.zeros(1, cfg., cfg.embed_dim))
    self.encoders = nn.ModuleList([encoder(cfg) for _ in range(cfg.depth)])
    self.norm = LayerNorm(cfg.embed_dim)
    self.head = nn.Linear(cfg.embed_dim, cfg.num_classes)

  def forward(self, x):
    # Giả sử x có độ dài thay đổi, cắt pos_embed cho khớp
    if x.shape[1] > self.pos_embed.shape[1]:
      x = x[:, :self.pos_embed.shape[1], :]
    # Cộng Positional Embedding
    x = x + self.pos_embed[:, :x.shape[1], :]

    for encoder in self.encoders:
      x = encoder(x)

    x = self.norm(x)
    x = x.mean(dim = 1) # Global Average Pooling
    x = self.head(x)
    return x


Overwriting core.py


In [6]:
# --- SYSTEM SIMULATION ---
import torch
from config import TinyConfig
from embeddings import PatchEmbed, AudioEmbed
from core import tinyTransformer

# 1. Khởi tạo Cấu hình
cfg = TinyConfig()
print(f"✅ Config loaded. Embed Dim: {cfg.embed_dim}, Heads: {cfg.num_heads}, Depth: {cfg.depth}")

# 2. Giả lập dữ liệu đầu vào
# Giả sử ảnh 64x64, 3 kênh màu (RGB)
dummy_img = torch.randn(1, 3, 64, 64)
# Giả sử audio feature (MFCC) có 64 khung thời gian, 40 đặc trưng
dummy_audio = torch.randn(1, 64, 40)

print(f"\n📸 Input Image Shape: {dummy_img.shape}")
print(f"Fnput Audio Shape: {dummy_audio.shape}")

# 3. Test Embeddings
patch_embed = PatchEmbed(cfg)
img_emb = patch_embed(dummy_img)
print(f"-> After PatchEmbed: {img_emb.shape}") # Mong đợi: [1, 64, 64]

audio_embed = AudioEmbed(cfg)
aud_emb = audio_embed(dummy_audio)
print(f"-> After AudioEmbed: {aud_emb.shape}") # Mong đợi: [1, 64, 64]

# 4. Test Core (Tiny Transformer)
model = tinyTransformer(cfg)
print("\n🤖 Model Initialized. Running Forward pass...")

# Chạy thử với Embedding ảnh
try:
    output = model(img_emb)
    print(f"🎉 Final Output Shape: {output.shape}")
    # Mong đợi: [1, 10] (Vì num_classes = 10)

    # Kiểm tra xem có chạy được backprop không (giả lập train)
    loss = output.sum()
    loss.backward()
    print("✅ Backward pass successful (Gradients computed).")
except Exception as e:
    print(f"❌ Error during forward/backward pass: {e}")

✅ Config loaded. Embed Dim: 64, Heads: 2, Depth: 3

📸 Input Image Shape: torch.Size([1, 3, 64, 64])
Fnput Audio Shape: torch.Size([1, 64, 40])
-> After PatchEmbed: torch.Size([1, 64, 64])
-> After AudioEmbed: torch.Size([1, 64, 64])

🤖 Model Initialized. Running Forward pass...
🎉 Final Output Shape: torch.Size([1, 10])
✅ Backward pass successful (Gradients computed).
